# CP4 · Notebook 06 — Compounding error en vivo

## Objetivo

Desplegar el policy BC entrenado **en el entorno cerrado** (closed-loop) y observar la **divergencia** que en P4 fue concepto.

**Tiempo**: ~10 min. Es el notebook **más importante** del CP.

In [ ]:
import numpy as np, json
from pathlib import Path
import torch, torch.nn as nn
import gymnasium as gym
import highway_env
import matplotlib.pyplot as plt

OUT = Path('../outputs')

class PilotNetMini(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 24, 5, stride=2), nn.ReLU(),
            nn.Conv2d(24, 36, 5, stride=2), nn.ReLU(),
            nn.Conv2d(36, 48, 3, stride=2), nn.ReLU(),
            nn.Conv2d(48, 64, 3), nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 6 * 6, 100), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(100, 1), nn.Tanh())
    def forward(self, x): return self.head(self.conv(x)).squeeze(-1)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = PilotNetMini().to(device)
model.load_state_dict(torch.load(OUT / '04_pilotnet.pt', map_location=device))
model.eval()
print('modelo cargado')

## 1. Crear entorno para rollout

Usamos el mismo entorno que el dataset pero con **una semilla distinta** para evitar memorización trivial.

In [ ]:
def make_env(seed=9999, ood=False):
    config = {
        'observation': {
            'type': 'GrayscaleObservation',
            'observation_shape': (84, 84), 'stack_size': 3,
            'weights': [0.2989, 0.5870, 0.1140], 'scaling': 1.75,
        },
        'action': {'type': 'ContinuousAction', 'longitudinal': False, 'lateral': True,
                   'steering_range': [-np.pi/4, np.pi/4]},
        'lanes_count': 3 if ood else 4,
        'vehicles_count': 35 if ood else 20,
        'duration': 60, 'policy_frequency': 5, 'simulation_frequency': 15,
    }
    env = gym.make('highway-v0', config=config)
    env.reset(seed=seed)
    return env

env = make_env(seed=9999, ood=False)
print('entorno creado')

## 2. Rollout — policy BC vs expert

Ejecutamos **el policy BC entrenado** durante 200 steps y grabamos la trayectoria (x, y del ego-vehicle). En paralelo, ejecutamos un **expert heurístico** (el mismo que generó el dataset) sobre **el mismo entorno con misma semilla** → la trayectoria del expert es nuestro "oracle".

In [ ]:
def predict_steer(obs):
    """obs es (3, 84, 84) según convención highway-env GrayscaleObservation.
    El modelo espera (3, 84, 84) ya en el orden de canales (= 3 frames stacked)."""
    x = obs.astype(np.float32) / 255.0
    x_t = torch.from_numpy(x[None, ...]).to(device)  # (1, 3, 84, 84)
    with torch.no_grad():
        return float(model(x_t).cpu().numpy()[0])

def expert_policy_visual(env):
    """Expert simplificado: lee posición ego del entorno interno y aplica heurística.
    Para el rollout solo necesitamos su steering en cada step."""
    # Acceso al ego vehicle
    ego = env.unwrapped.vehicle
    ego_lateral = ego.position[1]  # y absoluta del ego
    # Vecinos cercanos
    front_threat = False
    for v in env.unwrapped.road.vehicles:
        if v is ego: continue
        dx = v.position[0] - ego.position[0]
        dy = v.position[1] - ego.position[1]
        if 0 < dx < 30 and abs(dy) < 2:
            front_threat = True
            break
    if not front_threat:
        target_y = 4 * round(ego_lateral / 4)  # carriles de 4 m
        return float(np.clip(-0.3 * (ego_lateral - target_y), -1.0, 1.0))
    return -0.6 if ego_lateral >= 0 else 0.6

def run_rollout(steer_fn, seed=9999, n_steps=200, ood=False):
    env = make_env(seed=seed, ood=ood)
    obs, _ = env.reset(seed=seed)
    xs, ys, steers = [], [], []
    for _ in range(n_steps):
        steer = steer_fn(obs) if callable(steer_fn) else steer_fn
        # Posición ego
        xs.append(env.unwrapped.vehicle.position[0])
        ys.append(env.unwrapped.vehicle.position[1])
        steers.append(steer)
        action = np.array([steer], dtype=np.float32)
        obs, _, done, trunc, _ = env.step(action)
        if done or trunc: break
    env.close()
    return np.array(xs), np.array(ys), np.array(steers)

print('Rollout BC...')
bc_x, bc_y, bc_s = run_rollout(predict_steer, seed=9999, n_steps=200)

print('Rollout EXPERT (mismo seed)...')
# Truco: usamos el env nativo del expert porque necesita acceso a state interno
exp_env = make_env(seed=9999)
exp_obs, _ = exp_env.reset(seed=9999)
exp_xs, exp_ys, exp_ss = [], [], []
for _ in range(200):
    steer = expert_policy_visual(exp_env)
    exp_xs.append(exp_env.unwrapped.vehicle.position[0])
    exp_ys.append(exp_env.unwrapped.vehicle.position[1])
    exp_ss.append(steer)
    exp_obs, _, done, trunc, _ = exp_env.step(np.array([steer], dtype=np.float32))
    if done or trunc: break
exp_env.close()
exp_x, exp_y, exp_s = map(np.array, (exp_xs, exp_ys, exp_ss))

print(f'BC rollout:     {len(bc_x)} steps')
print(f'Expert rollout: {len(exp_x)} steps')

## 3. Plot del closed-loop — el momento de la verdad

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(exp_x, exp_y, color='#4a6fa5', label='Expert (oracle)', linewidth=2)
axes[0].plot(bc_x,  bc_y,  color='#DA4544', label='BC policy entrenado', linewidth=2)
axes[0].set_xlabel('x (m)')
axes[0].set_ylabel('y (m) — lateral')
axes[0].set_title('Trayectorias closed-loop (mismo seed, mismo entorno)')
axes[0].legend(); axes[0].grid(alpha=0.3)
axes[0].axhline(0, color='black', alpha=0.3, linestyle=':')

# Steering aplicado en cada step
axes[1].plot(exp_s, color='#4a6fa5', label='Expert steering', alpha=0.7)
axes[1].plot(bc_s,  color='#DA4544', label='BC steering',     alpha=0.7)
axes[1].set_xlabel('step')
axes[1].set_ylabel('steering [-1, 1]')
axes[1].set_title('Steering aplicado por step')
axes[1].legend(); axes[1].grid(alpha=0.3)
axes[1].axhline(0, color='black', alpha=0.3, linestyle=':')

plt.tight_layout(); plt.savefig(OUT / '06_closed_loop.png', dpi=100, bbox_inches='tight'); plt.show()

## 4. Métrica de divergencia

Tomamos los **primeros 100 steps** (donde la longitud coincide) y medimos la **distancia lateral media** entre BC y expert.

In [ ]:
n_common = min(len(bc_y), len(exp_y))
lateral_diff = np.abs(bc_y[:n_common] - exp_y[:n_common])

print(f'Steps comparables:        {n_common}')
print(f'Distancia lateral media:  {lateral_diff.mean():.2f} m')
print(f'Distancia lateral max:    {lateral_diff.max():.2f} m')
print(f'Step donde se aleja >1m:  ', end='')
above = np.where(lateral_diff > 1.0)[0]
print(above[0] if len(above) else '> nunca durante el rollout (modelo aguanta!)')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lateral_diff, color='#DA4544', linewidth=2)
ax.axhline(1.0, color='orange', linestyle='--', label='1 m')
ax.axhline(2.0, color='red', linestyle='--', label='2 m (medio carril)')
ax.set_xlabel('step'); ax.set_ylabel('|y_BC - y_expert| (m)')
ax.set_title('Compounding error: divergencia lateral BC vs expert')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(OUT / '06_divergence.png', dpi=100, bbox_inches='tight'); plt.show()

## 5. Lección clave

Lo que acabas de ver:

- Aunque MSE en val_in fuera **bajo** (notebook 05), el **closed-loop diverge**.
- La divergencia **crece con el tiempo** — exactamente el patrón de compounding error.
- En **un sistema real** este policy nunca pasaría QA — saldría del carril en segundos.

**Por qué Tesla puede y nosotros no**:
1. Tesla tiene **5M coches** → la distribución de deployment cae casi siempre en training.
2. Tesla puede **iterar continuamente** (data → train → deploy → data) — DAgger online a escala mundial.
3. Tesla combina BC con **planning module clásico** que recupera de fallos del BC.

Apunta esto para el entregable. Ve a `07_preguntas.md`.